# VisionBridge V3 checkpoint evaluation (Colab)

**Purpose:** evaluate the checkpoint currently in the VisionBridge repository without training, fine-tuning, or changing model weights.

This notebook:
1. Clones the current repository and creates an isolated Python 3.12 environment.
2. Reuses compatible prepared landmarks if they already exist; otherwise downloads the pinned RealSign archive and MediaPipe Hand Landmarker asset, verifies their hashes, and prepares the splits.
3. Checks that the checkpoint matches the active V3 contract (126D input, A-Z classes, 64D embedding).
4. Evaluates the **test split once** using the repository's evaluation script.
5. Produces per-letter precision/recall/F1, confusion matrices, common error pairs, a hash-bound JSON report, and a downloadable ZIP bundle.

**Important:** this is an evaluation-only notebook. Do not use the test result to choose hyperparameters or repeatedly compare candidate models. Use validation data for model selection. A fresh runtime may need to download about 657 MB and run landmark extraction, which can take a while on CPU.

For fastest inference, select a Colab GPU runtime before starting. Landmark extraction itself is not a GPU training job.

In [ ]:
# Cell 1 — Clean workspace + Python 3.12 env + deps
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time

ROOT = Path("/content")
os.chdir(ROOT)

REPO = ROOT / "VisionBridge"
VENV = ROOT / "visionbridge_train_env"
PYTHON = VENV / "bin" / "python"

# Prevent Colab's matplotlib_inline backend from breaking MediaPipe imports
# inside the isolated venv (matplotlib_inline is not installed there).
os.environ["MPLBACKEND"] = "Agg"


def run(cmd, env=None, check=True, cwd=None):
    cmd = [str(value) for value in cmd]
    print("$", " ".join(cmd), flush=True)
    result = subprocess.run(
        cmd,
        check=False,
        env=env,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"  {' '.join(cmd)}\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )
    return result


def run_retry(cmd, attempts=3, env=None, cwd=None):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            print(f"Attempt {attempt}/{attempts}", flush=True)
            return run(cmd, env=env, cwd=cwd)
        except Exception as exc:
            last_error = exc
            print("Attempt failed:", repr(exc), flush=True)
            if attempt < attempts:
                time.sleep(5 * attempt)
    raise RuntimeError(
        f"Command failed after {attempts} attempts: " + " ".join(str(x) for x in cmd)
    ) from last_error


def make_venv_env():
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO / "backend")
    env["MPLBACKEND"] = "Agg"
    env["PYTHONUNBUFFERED"] = "1"
    # Avoid leaking Colab notebook backend into child matplotlib
    env.pop("MPLCONFIGDIR", None)
    return env


if REPO.exists():
    print("Removing previous /content/VisionBridge workspace...")
    shutil.rmtree(REPO)

if VENV.exists():
    print("Removing previous training environment...")
    shutil.rmtree(VENV)

print("\nCloning VisionBridge...")
run_retry(
    [
        "git", "clone", "--depth", "1",
        "https://github.com/BharathWaj-K-R/VisionBridge.git",
        str(REPO),
    ],
    attempts=3,
)
if not (REPO / ".git").is_dir():
    raise RuntimeError("VisionBridge clone did not produce a valid Git repository.")
print("VisionBridge source workspace: PASS")

print("\nInstalling uv...")
run_retry(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "--retries", "5", "--timeout", "120", "uv",
    ],
    attempts=3,
)
UV = [sys.executable, "-m", "uv"]

print("\nEnsuring Python 3.12 is available...")
run_retry([*UV, "python", "install", "3.12"], attempts=3)

print("\nCreating isolated training environment...")
run([*UV, "venv", "--python", "3.12", str(VENV)])
if not PYTHON.is_file():
    raise RuntimeError(f"Training Python was not created: {PYTHON}")
print("Training Python:", PYTHON)

# Bootstrap pip inside the venv (needed for torch index installs)
print("\nBootstrapping pip in the training venv...")
run_retry([*UV, "pip", "install", "--python", str(PYTHON), "pip", "setuptools", "wheel"], attempts=3)

try:
    gpu_probe = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    has_gpu = gpu_probe.returncode == 0
except FileNotFoundError:
    # Colab's GPU runtime ships nvidia-smi; plain CI/CPU runners may not.
    has_gpu = False
print("GPU available:", has_gpu)
print("Training device will be selected by PyTorch after installation.")

print("\nInstalling NumPy 2.1.3 + MediaPipe 0.10.35...")
run_retry(
    [
        *UV, "pip", "install", "--python", str(PYTHON), "--reinstall",
        "numpy==2.1.3", "mediapipe==0.10.35",
    ],
    attempts=3,
)

# PyTorch: try CUDA indexes first, then CPU. Use --extra-index-url so PyPI
# deps stay available, and re-pin numpy afterward so torch cannot clobber it.
PIP = [str(PYTHON), "-m", "pip"]
torch_candidates = []
if has_gpu:
    torch_candidates.extend([
        ("CUDA 12.8", "https://download.pytorch.org/whl/cu128"),
        ("CUDA 12.6", "https://download.pytorch.org/whl/cu126"),
    ])
torch_candidates.append(("CPU", "https://download.pytorch.org/whl/cpu"))

torch_installed = False
torch_errors = []
child_env = make_venv_env()

for label, index_url in torch_candidates:
    print(f"\nAttempting PyTorch 2.9.0 ({label})...")
    result = subprocess.run(
        [
            *PIP, "install", "--disable-pip-version-check", "--no-cache-dir",
            "--retries", "5", "--timeout", "120",
            "torch==2.9.0",
            "--extra-index-url", index_url,
        ],
        check=False,
        text=True,
        capture_output=True,
        env=child_env,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)

    if result.returncode == 0:
        # Re-pin numpy in case a dependency shifted it
        subprocess.run(
            [*PIP, "install", "--disable-pip-version-check", "--quiet", "numpy==2.1.3"],
            check=False, text=True, capture_output=True, env=child_env,
        )
        import_test = subprocess.run(
            [
                str(PYTHON), "-c",
                (
                    "import os; os.environ['MPLBACKEND']='Agg'; "
                    "import torch; import numpy as np; "
                    "print('torch_version=' + torch.__version__); "
                    "print('numpy_version=' + np.__version__); "
                    "print('cuda_available=' + str(torch.cuda.is_available())); "
                    "assert torch.__version__.split('+')[0] == '2.9.0'; "
                    "assert np.__version__ == '2.1.3'"
                ),
            ],
            check=False,
            text=True,
            capture_output=True,
            env=child_env,
        )
        if import_test.stdout:
            print(import_test.stdout, end="")
        if import_test.stderr:
            print(import_test.stderr, end="", file=sys.stderr)
        if import_test.returncode == 0:
            torch_installed = True
            print(f"PyTorch 2.9.0 ({label}): PASS")
            break

    error_text = result.stderr or result.stdout or "unknown installation error"
    torch_errors.append(f"{label}: {error_text}")
    print(f"PyTorch {label} installation did not succeed; trying the next supported build.")

if not torch_installed:
    raise RuntimeError(
        "No supported PyTorch 2.9.0 build could be installed.\n\n" + "\n\n".join(torch_errors)
    )

ENV = make_venv_env()
version_check = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
import sys
import torch
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

assert sys.version_info[:2] == (3, 12), sys.version_info
assert np.__version__ == "2.1.3", np.__version__
assert mp.__version__ == "0.10.35", mp.__version__
print("MediaPipe Tasks API:", vision.HandLandmarker.__name__)
print("Training environment: PASS")
'''
run([str(PYTHON), "-c", version_check], env=ENV)
print("\nCell 1 complete.")

In [ ]:
# Cell 2 — Reuse only a compatible prepared dataset; otherwise rebuild it
from pathlib import Path
import json

DATA_DIR = Path("/content/visionbridge_letter_data")
required = [
    DATA_DIR / "train.npz",
    DATA_DIR / "val.npz",
    DATA_DIR / "test.npz",
    DATA_DIR / "labels.json",
    DATA_DIR / "duplicate_report.json",
    DATA_DIR / "train_manifest.jsonl",
    DATA_DIR / "val_manifest.jsonl",
    DATA_DIR / "test_manifest.jsonl",
]

def prepared_data_is_compatible():
    if not all(path.is_file() for path in required):
        return False
    try:
        metadata = json.loads((DATA_DIR / "labels.json").read_text(encoding="utf-8"))
        duplicates = json.loads((DATA_DIR / "duplicate_report.json").read_text(encoding="utf-8"))
        return (
            metadata.get("labels") == list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
            and metadata.get("preprocessing_version") == "two-hand-wrist-scale-v1"
            and metadata.get("landmarker", {}).get("runtime") == "mediapipe-hand-landmarker-0.10.35"
            and metadata.get("split_policy", {}).get("test_source_left_untouched") is True
            and not duplicates.get("train_val_overlap_hashes")
            and not duplicates.get("train_test_overlap_hashes")
            and not duplicates.get("val_test_overlap_hashes")
        )
    except (OSError, ValueError, TypeError):
        return False

USE_EXISTING_PREPARED_DATA = prepared_data_is_compatible()
if USE_EXISTING_PREPARED_DATA:
    print("Compatible prepared dataset found. Existing NPZ files will be validated before evaluation.")
else:
    print("No compatible prepared dataset found. The notebook will rebuild it from the pinned RealSign source.")


In [ ]:
if not USE_EXISTING_PREPARED_DATA:
    from pathlib import Path
    import hashlib
    import time
    import urllib.request
    
    HAND_MODEL_URL = (
        "https://storage.googleapis.com/mediapipe-models/"
        "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    )
    HAND_MODEL_PATH = Path("/content/hand_landmarker.task")
    HAND_MODEL_SIZE = 7_819_105
    HAND_MODEL_SHA256 = "fbc2a30080c3c557093b5ddfc334698132eb341044ccee322ccf8bcf3607cde1"
    MAX_ATTEMPTS = 4
    
    if HAND_MODEL_PATH.exists():
        HAND_MODEL_PATH.unlink()
    PARTIAL = HAND_MODEL_PATH.with_suffix(".task.part")
    if PARTIAL.exists():
        PARTIAL.unlink()
    
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            print(f"Downloading MediaPipe Hand Landmarker (attempt {attempt}/{MAX_ATTEMPTS})...")
            request = urllib.request.Request(
                HAND_MODEL_URL,
                headers={"User-Agent": "VisionBridge-V3-training/1.0"},
            )
            with urllib.request.urlopen(request, timeout=180) as response, PARTIAL.open("wb") as output:
                while True:
                    chunk = response.read(8 * 1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
            PARTIAL.replace(HAND_MODEL_PATH)
            last_error = None
            break
        except Exception as exc:
            last_error = exc
            print("Download attempt failed:", repr(exc))
            if PARTIAL.exists():
                PARTIAL.unlink()
            if attempt < MAX_ATTEMPTS:
                time.sleep(3 * attempt)
    
    if last_error is not None:
        raise RuntimeError(
            f"MediaPipe Hand Landmarker download failed after {MAX_ATTEMPTS} attempts."
        ) from last_error
    
    size = HAND_MODEL_PATH.stat().st_size
    digest = hashlib.sha256()
    with HAND_MODEL_PATH.open("rb") as file:
        while True:
            chunk = file.read(8 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    sha256 = digest.hexdigest()
    
    if size != HAND_MODEL_SIZE:
        raise RuntimeError(f"Hand model size mismatch: expected {HAND_MODEL_SIZE}, got {size}")
    if sha256 != HAND_MODEL_SHA256:
        raise RuntimeError(
            f"Hand model SHA-256 mismatch: expected {HAND_MODEL_SHA256}, got {sha256}"
        )
    
    print("Hand model bytes:", size)
    print("Hand model SHA-256:", sha256)
    print("Hand model integrity: PASS")
else:
    print("Reusing verified prepared data; skipped Hand Landmarker download.")


In [ ]:
if not USE_EXISTING_PREPARED_DATA:
    import os
    import subprocess
    import sys
    from pathlib import Path
    
    PYTHON = Path("/content/visionbridge_train_env/bin/python")
    MODEL = Path("/content/hand_landmarker.task")
    
    if not PYTHON.exists():
        raise RuntimeError("Training environment is missing. Run Cell 1.")
    if not MODEL.exists():
        raise RuntimeError("MediaPipe hand model is missing. Run Cell 2.")
    
    ENV = os.environ.copy()
    ENV["MPLBACKEND"] = "Agg"
    
    smoke = r'''
    import os
    os.environ["MPLBACKEND"] = "Agg"
    import sys
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    model_path = sys.argv[1]
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        num_hands=2,
    )
    detector = vision.HandLandmarker.create_from_options(options)
    detector.close()
    print("Hand Landmarker initialization: PASS")
    '''
    
    result = subprocess.run(
        [str(PYTHON), "-c", smoke, str(MODEL)],
        check=False, env=ENV, text=True, capture_output=True,
    )
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "Hand Landmarker smoke test failed.\n"
            f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        )
    print("Cell 3 complete.")
else:
    print("Reusing verified prepared data; skipped Hand Landmarker smoke test.")


In [ ]:
if not USE_EXISTING_PREPARED_DATA:
    from pathlib import Path
    import hashlib
    import shutil
    import time
    import urllib.request
    import zipfile
    
    DATASET_DIR = Path("/content/RealSign")
    ZIP_PATH = Path("/content/RealSign_Dataset.zip")
    SOURCE_REPOSITORY = "https://github.com/RealSign62/RealSign-Indian-Sign-Language-Dataset"
    SOURCE_COMMIT = "17c51dcc158b7b6359b7c6667edc51c7f148f3f4"
    SOURCE_URL = (
        "https://media.githubusercontent.com/media/"
        "RealSign62/RealSign-Indian-Sign-Language-Dataset/"
        f"{SOURCE_COMMIT}/Dataset.zip"
    )
    EXPECTED_SIZE = 656_689_688
    EXPECTED_SHA256 = "008cae248e346b8c31fbbea057fcc3f69c6909d29a88e6bb1fb0369f528de2b5"
    MAX_ATTEMPTS = 5
    
    if DATASET_DIR.exists():
        print("Removing previous RealSign extraction...")
        shutil.rmtree(DATASET_DIR)
    if ZIP_PATH.exists():
        print("Removing previous RealSign archive...")
        ZIP_PATH.unlink()
    PARTIAL_PATH = ZIP_PATH.with_suffix(".zip.part")
    if PARTIAL_PATH.exists():
        PARTIAL_PATH.unlink()
    
    print("RealSign source:", SOURCE_REPOSITORY)
    print("Direct download URL:", SOURCE_URL)
    
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            print(f"\nDownloading RealSign Dataset.zip (attempt {attempt}/{MAX_ATTEMPTS})...")
            request = urllib.request.Request(
                SOURCE_URL,
                headers={"User-Agent": "VisionBridge-V3-training/1.0"},
            )
            with urllib.request.urlopen(request, timeout=300) as response, PARTIAL_PATH.open("wb") as output:
                downloaded = 0
                while True:
                    chunk = response.read(8 * 1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
                    downloaded += len(chunk)
                    if downloaded >= 64 * 1024 * 1024 and downloaded % (64 * 1024 * 1024) < 8 * 1024 * 1024:
                        print(f"Downloaded {downloaded:,} bytes", flush=True)
            PARTIAL_PATH.replace(ZIP_PATH)
            last_error = None
            break
        except Exception as exc:
            last_error = exc
            print("RealSign download attempt failed:", repr(exc))
            if PARTIAL_PATH.exists():
                PARTIAL_PATH.unlink()
            if attempt < MAX_ATTEMPTS:
                time.sleep(5 * attempt)
    
    if last_error is not None:
        raise RuntimeError(
            f"RealSign Dataset.zip download failed after {MAX_ATTEMPTS} attempts."
        ) from last_error
    
    size = ZIP_PATH.stat().st_size
    if size != EXPECTED_SIZE:
        raise RuntimeError(f"RealSign archive size mismatch: expected {EXPECTED_SIZE}, got {size}")
    
    sha256 = hashlib.sha256()
    with ZIP_PATH.open("rb") as file:
        while True:
            chunk = file.read(8 * 1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)
    digest = sha256.hexdigest()
    if digest != EXPECTED_SHA256:
        raise RuntimeError(
            f"RealSign archive SHA-256 mismatch: expected {EXPECTED_SHA256}, got {digest}"
        )
    
    if not zipfile.is_zipfile(ZIP_PATH):
        raise RuntimeError("Downloaded RealSign file is not a valid ZIP archive.")
    
    with zipfile.ZipFile(ZIP_PATH) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"ZIP integrity check failed at member: {bad_member}")
        archive.extractall(DATASET_DIR)
    
    expected = {"Training": False, "Validation": False, "Testing": False}
    for path in DATASET_DIR.rglob("*"):
        if not path.is_dir():
            continue
        normalized = path.name.strip().lower()
        if normalized in {"training", "training (a-z)"}:
            expected["Training"] = True
        elif normalized in {"validation", "validation (a-z)", "val"}:
            expected["Validation"] = True
        elif normalized in {"testing", "testing (a-z)", "test"}:
            expected["Testing"] = True
    
    missing = [name for name, exists in expected.items() if not exists]
    if missing:
        raise RuntimeError(
            "RealSign extraction is missing expected split folders: " + ", ".join(missing)
        )
    
    print("\nRealSign archive bytes:", size)
    print("RealSign archive SHA-256:", digest)
    print("Training / Validation / Testing folders: PASS")
    print("RealSign direct download + extraction: PASS")
else:
    print("Reusing verified prepared data; skipped RealSign download and extraction.")


In [ ]:
if not USE_EXISTING_PREPARED_DATA:
    import os
    import subprocess
    import sys
    from pathlib import Path
    
    REPO = Path("/content/VisionBridge")
    PYTHON = Path("/content/visionbridge_train_env/bin/python")
    ENV = os.environ.copy()
    ENV["PYTHONPATH"] = str(REPO / "backend")
    ENV["MPLBACKEND"] = "Agg"
    ENV["PYTHONUNBUFFERED"] = "1"
    
    out_dir = Path("/content/visionbridge_letter_data")
    if not PYTHON.exists():
        raise RuntimeError("Training Python is missing. Run Cell 1.")
    if not Path("/content/RealSign").is_dir():
        raise RuntimeError("RealSign dataset is missing. Run Cell 4.")
    if not Path("/content/hand_landmarker.task").is_file():
        raise RuntimeError("Hand model is missing. Run Cell 2.")
    
    if out_dir.exists():
        import shutil
        shutil.rmtree(out_dir)
    
    command = [
        str(PYTHON),
        str(REPO / "backend/scripts/prepare_letter_dataset.py"),
        "--input-root", "/content/RealSign",
        "--output-dir", str(out_dir),
        "--validation-ratio", "0.20",
        "--seed", "42",
        "--hand-model-path", "/content/hand_landmarker.task",
    ]
    
    print("Running landmark preparation (this is the slow step)...", flush=True)
    result = subprocess.run(command, check=False, env=ENV, text=True, capture_output=True)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    
    required = [
        out_dir / "train.npz",
        out_dir / "val.npz",
        out_dir / "test.npz",
        out_dir / "labels.json",
        out_dir / "duplicate_report.json",
    ]
    if result.returncode != 0:
        raise RuntimeError(
            f"prepare_letter_dataset.py failed (exit {result.returncode})\n"
            f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        )
    missing_outputs = [str(path) for path in required if not path.is_file()]
    if missing_outputs:
        raise RuntimeError(
            "Landmark preparation finished but required outputs are missing:\n"
            + "\n".join(missing_outputs)
        )
    print("Landmark preparation: PASS")
else:
    print("Reusing verified prepared data; skipped landmark preparation.")


In [ ]:
# Cell 8 — Validate all prepared splits and leakage guards
from pathlib import Path
import json
import numpy as np

DATA_DIR = Path("/content/visionbridge_letter_data")
metadata = json.loads((DATA_DIR / "labels.json").read_text(encoding="utf-8"))
duplicate_report = json.loads((DATA_DIR / "duplicate_report.json").read_text(encoding="utf-8"))

assert metadata["labels"] == list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"), "Dataset label order must be A-Z."
assert metadata["preprocessing_version"] == "two-hand-wrist-scale-v1", "Unexpected preprocessing version."
assert metadata["landmarker"]["runtime"] == "mediapipe-hand-landmarker-0.10.35", "Unexpected landmark runtime."
assert metadata["split_policy"]["test_source_left_untouched"] is True, "Test split must remain the source testing split."

for split in ("train", "val", "test"):
    with np.load(DATA_DIR / f"{split}.npz", allow_pickle=False) as data:
        x = data["x"]
        y = data["y"]
    assert x.ndim == 2 and x.shape[1] == 126, f"{split}: expected [N,126], got {x.shape}"
    assert y.ndim == 1 and len(x) == len(y) and len(y) > 0, f"{split}: invalid labels."
    assert np.isfinite(x).all(), f"{split}: features contain NaN/Inf."
    assert y.min() >= 0 and y.max() < 26, f"{split}: label index outside A-Z."
    counts = np.bincount(y, minlength=26)
    assert len(counts) == 26 and np.all(counts > 0), f"{split}: missing one or more A-Z classes."
    manifest_count = sum(1 for _ in (DATA_DIR / f"{split}_manifest.jsonl").open(encoding="utf-8"))
    assert manifest_count == len(y), f"{split}: manifest count differs from NPZ count."
    print(f"{split}: samples={len(y):,}; feature_shape={x.shape}; min/class={counts.min()}; max/class={counts.max()}")

assert not duplicate_report["train_val_overlap_hashes"], "Train/validation duplicate leakage detected."
assert not duplicate_report["train_test_overlap_hashes"], "Train/test duplicate leakage detected."
assert not duplicate_report["val_test_overlap_hashes"], "Validation/test duplicate leakage detected."
print("\nDataset contract and exact-image duplicate guards: PASS")


In [ ]:
# Cell 9 — Validate exact checkpoint and V3 model contract
import hashlib
import json
import os
import subprocess
from pathlib import Path

CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"
if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise FileNotFoundError(f"Checkpoint missing or empty: {CHECKPOINT}")

ENV = make_venv_env()
script = r"""
import hashlib, json, sys
from pathlib import Path
import torch
from app.models.letter_model import (
    INPUT_DIM, EMBEDDING_DIM, NUM_CLASSES, LETTER_LABELS,
    MODEL_VERSION, PREPROCESSING_VERSION, LANDMARK_RUNTIME, load_checkpoint
)
path = Path(sys.argv[1])
model = load_checkpoint(path)
assert model.input_dim == INPUT_DIM == 126
assert model.embedding_dim == EMBEDDING_DIM == 64
assert model.num_classes == NUM_CLASSES == 26
assert tuple(model.labels) == tuple(LETTER_LABELS) == tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
sample = torch.zeros(4, 126)
with torch.inference_mode():
    logits = model(sample)
    embedding = model.embed(sample)
assert tuple(logits.shape) == (4, 26)
assert tuple(embedding.shape) == (4, 64)
assert torch.isfinite(logits).all() and torch.isfinite(embedding).all()
digest = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps({
    "checkpoint": str(path),
    "sha256": digest,
    "bytes": path.stat().st_size,
    "model_version": MODEL_VERSION,
    "preprocessing_version": PREPROCESSING_VERSION,
    "landmark_runtime": LANDMARK_RUNTIME,
    "input_dim": model.input_dim,
    "hidden_dim": model.hidden_dim,
    "embedding_dim": model.embedding_dim,
    "num_classes": model.num_classes,
    "parameters": sum(p.numel() for p in model.parameters()),
    "device_cuda_available": torch.cuda.is_available()
}, indent=2))
"""
result = subprocess.run([str(PYTHON), "-c", script, str(CHECKPOINT)], env=ENV, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Checkpoint validation failed. Do not continue to evaluation.")
checkpoint_info = json.loads(result.stdout.strip())
CHECKPOINT_SHA256 = checkpoint_info["sha256"]
print("Checkpoint V3 contract: PASS")


In [ ]:
# Cell 10 — Evaluate the untouched test split (no training)
import os
import subprocess
from pathlib import Path

OUTPUT_JSON = Path("/content/visionbridge_v3_evaluation.json")
ENV = make_venv_env()
command = [
    str(PYTHON),
    str(REPO / "backend/app/training/evaluate_letter_base.py"),
    "--checkpoint", str(CHECKPOINT),
    "--data-dir", str(DATA_DIR),
    "--split", "test",
    "--batch-size", "128",
    "--output-json", str(OUTPUT_JSON),
]
print("$", " ".join(command), flush=True)
result = subprocess.run(command, env=ENV, text=True, capture_output=True)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="")
if result.returncode != 0:
    raise RuntimeError(f"Evaluation failed with exit code {result.returncode}.")
if not OUTPUT_JSON.is_file():
    raise RuntimeError("Evaluator did not create the JSON report.")
print("\nEvaluation report saved:", OUTPUT_JSON)


In [ ]:
# Cell 11 — Per-letter metrics, confusion analysis, and plots
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

raw_report = json.loads(Path("/content/visionbridge_v3_evaluation.json").read_text(encoding="utf-8"))
labels = raw_report["labels"]
cm = np.asarray(raw_report["confusion_matrix"], dtype=np.int64)
assert cm.shape == (26, 26) and labels == list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

tp = np.diag(cm)
support = cm.sum(axis=1)
predicted_count = cm.sum(axis=0)
precision = np.divide(tp, predicted_count, out=np.zeros_like(tp, dtype=float), where=predicted_count != 0)
recall = np.divide(tp, support, out=np.zeros_like(tp, dtype=float), where=support != 0)
f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp, dtype=float), where=(precision + recall) != 0)
per_letter = {}
for i, letter in enumerate(labels):
    per_letter[letter] = {
        "support": int(support[i]),
        "predicted_count": int(predicted_count[i]),
        "precision": float(precision[i]),
        "recall": float(recall[i]),
        "f1": float(f1[i]),
    }

detailed = {
    "checkpoint": raw_report["checkpoint"],
    "checkpoint_sha256": raw_report["checkpoint_sha256"],
    "split": raw_report["split"],
    "samples": int(raw_report["samples"]),
    "overall_accuracy": float(raw_report["overall_accuracy"]),
    "macro_recall": float(recall.mean()),
    "macro_precision": float(precision.mean()),
    "macro_f1": float(f1.mean()),
    "per_letter": per_letter,
}
detailed_path = Path("/content/visionbridge_v3_detailed_metrics.json")
detailed_path.write_text(json.dumps(detailed, indent=2), encoding="utf-8")

print("Overall test accuracy:", f"{raw_report['overall_accuracy']:.4%}")
print("Macro recall / balanced accuracy:", f"{recall.mean():.4%}")
print("Macro precision:", f"{precision.mean():.4%}")
print("Macro F1:", f"{f1.mean():.4%}")
print("Worst letter by recall:", min(per_letter, key=lambda label: per_letter[label]["recall"]),
      f"({min(x['recall'] for x in per_letter.values()):.4%})")

table = pd.DataFrame([
    {"Letter": label, **values}
    for label, values in per_letter.items()
]).set_index("Letter")
display(table.style.format({
    "precision": "{:.2%}", "recall": "{:.2%}", "f1": "{:.2%}"
}))

# Most frequent incorrect true -> predicted letter pairs.
errors = []
for i, true_letter in enumerate(labels):
    for j, predicted_letter in enumerate(labels):
        if i != j and cm[i, j] > 0:
            errors.append((int(cm[i, j]), true_letter, predicted_letter))
errors.sort(reverse=True)
print("\nMost frequent wrong predictions (true -> predicted):")
for count, truth, prediction in errors[:15]:
    print(f"{truth} -> {prediction}: {count}")

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("VisionBridge V3: Test Confusion Matrix")
ax.set_xlabel("Predicted letter")
ax.set_ylabel("True letter")
ax.set_xticks(np.arange(26), labels=labels)
ax.set_yticks(np.arange(26), labels=labels)
plt.setp(ax.get_xticklabels(), rotation=90)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
confusion_path = Path("/content/visionbridge_v3_confusion_matrix.png")
fig.savefig(confusion_path, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(labels, recall)
ax.axhline(raw_report["overall_accuracy"], linestyle="--", label="Overall accuracy")
ax.set_ylim(0, 1.02)
ax.set_title("Per-letter recall (same as per-letter accuracy)")
ax.set_xlabel("True letter")
ax.set_ylabel("Recall")
ax.legend()
fig.tight_layout()
per_letter_plot = Path("/content/visionbridge_v3_per_letter_recall.png")
fig.savefig(per_letter_plot, dpi=180, bbox_inches="tight")
plt.show()

# Compare repository reference only when it refers to the exact same checkpoint bytes.
reference_path = REPO / "visionbridge_letter_evaluation.json"
if reference_path.is_file():
    reference = json.loads(reference_path.read_text(encoding="utf-8"))
    if reference.get("checkpoint_sha256") == raw_report["checkpoint_sha256"]:
        print("\nRepository reference report matches this checkpoint.")
        print("Reference overall accuracy:", f"{reference['overall_accuracy']:.4%}")
    else:
        print("\nNOTE: Existing repository report uses a different checkpoint SHA-256.")
        print("Do not compare its accuracy directly with this run; this notebook's result is bound to the evaluated checkpoint hash.")

print("\nSaved detailed metrics:", detailed_path)
print("Saved confusion matrix:", confusion_path)
print("Saved per-letter recall plot:", per_letter_plot)


In [ ]:
# Cell 12 — Bundle evaluation evidence for download
import json
import zipfile
from pathlib import Path

OUTPUTS = [
    Path("/content/visionbridge_v3_evaluation.json"),
    Path("/content/visionbridge_v3_detailed_metrics.json"),
    Path("/content/visionbridge_v3_confusion_matrix.png"),
    Path("/content/visionbridge_v3_per_letter_recall.png"),
]
missing = [str(path) for path in OUTPUTS if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing evaluation outputs:\n" + "\n".join(missing))

bundle = Path("/content/visionbridge_v3_evaluation_bundle.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUTS:
        archive.write(path, arcname=path.name)

print("Evaluation evidence bundle:", bundle)
print("Included files:")
for path in OUTPUTS:
    print(" -", path.name, f"({path.stat().st_size:,} bytes)")
print("\nIn Colab's Files pane, right-click visionbridge_v3_evaluation_bundle.zip and choose Download.")
